# Speech-to-Text Model Comparison

**For this test, I am using a shorter audio clip from (freesound.org)[https://freesound.org/people/guidofm/sounds/705574/]. The clip was selected because it is shorter and has a faster processing and loading times.**

This notebook compares four popular open-source speech-to-text models for their transcription accuracy, speed, and ease of use. The models covered are:

- **Wav2Vec 2.0**: A highly popular model developed by Facebook AI, known for its robust performance on a wide range of tasks, including speech recognition. [Wav2Vec 2.0 Documentation](https://github.com/pytorch/fairseq/tree/main/examples/wav2vec).

- **Whisper**: Developed by OpenAI, this model is designed for multilingual speech recognition and supports a wide range of languages. [Whisper Documentation](https://github.com/openai/whisper).

- **VOSK**: A lightweight speech-to-text toolkit that supports offline processing and multiple languages. [VOSK Documentation](https://alphacephei.com/vosk/).

This notebook will evaluate these models to assess their performance on transcription tasks, including word accuracy and speaker differentiation. The choice of models was provided by [Gladia's blog](https://www.gladia.io/blog/best-open-source-speech-to-text-models).

In [1]:
#Load the wav file
audio_file = "/Users/laiscoletta/Desktop/DA_Final_Semester/Machine-Learning/Audio Files/audio_for_testing.wav"

## Test 1: Wav2Vec 2.0

In [2]:
#Reference code: https://www.kaggle.com/code/dikshabhati2002/speech-to-text-with-hugging-face , https://github.com/facebookresearch/fairseq/blob/main/examples/wav2vec/README.md
import transformers
import librosa
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from pyannote.audio import Pipeline
import numpy as np

# Load Wav2Vec 2.0 model and processor
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

# Load the pre-trained speaker diarization model from pyannote
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization@2.1")

def speech_to_text_wav2vec(audio_path, start, end):
    # Load audio using librosa with offset and duration to segment the audio
    speech, rate = librosa.load(audio_path, sr=16000, offset=start, duration=end - start)  # Resample to 16kHz
    
    # Preprocess the audio
    input_values = processor(speech, sampling_rate=rate, return_tensors="pt", padding=True).input_values
    
    # Perform speech-to-text
    with torch.no_grad():
        logits = model(input_values).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    
    # Decode the predicted IDs to text
    transcription = processor.batch_decode(predicted_ids)[0]
    return transcription

def diarize_and_transcribe(audio_file):
    # Perform speaker diarization using pyannote
    diarization = pipeline(audio_file)

    # Create speaker mapping and collect transcription segments
    speaker_mapping = {}
    speaker_id = 0
    segments = []

    # Iterate through diarization and align transcription with speaker segments
    for segment, _, speaker in diarization.itertracks(yield_label=True):
        # Create a mapping for speakers
        if speaker not in speaker_mapping:
            speaker_mapping[speaker] = f"Speaker {speaker_id + 1}"
            speaker_id += 1

        # Append the speaker, start, and end time for each segment
        segments.append({
            "speaker": speaker_mapping[speaker],
            "start": segment.start,
            "end": segment.end
        })

    # Transcribe and print the transcription for each speaker segment
    for segment in segments:
        transcription = speech_to_text_wav2vec(audio_file, segment["start"], segment["end"])
        print(f"[{segment['speaker']}] {transcription}")

# Run diarization and transcription
diarize_and_transcribe(audio_file)


objc[97389]: Class AVFFrameReceiver is implemented in both /Applications/anaconda3/lib/python3.12/site-packages/av/.dylibs/libavdevice.61.3.100.dylib (0x136bac0f8) and /Applications/anaconda3/lib/libavdevice.60.3.100.dylib (0x15634c0f8). One of the two will be used. Which one is undefined.
objc[97389]: Class AVFAudioReceiver is implemented in both /Applications/anaconda3/lib/python3.12/site-packages/av/.dylibs/libavdevice.61.3.100.dylib (0x136bac148) and /Applications/anaconda3/lib/libavdevice.60.3.100.dylib (0x15634c148). One of the two will be used. Which one is undefined.
/Applications/anaconda3/lib/python3.12/site-packages/pyannote/audio/core/io.py:43: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: 

Model was trained with pyannote.audio 0.0.1, yours is 3.1.1. Bad things might happen unless you revert pyannote.audio to 0.x.
Model was trained with torch 1.10.0+cu102, yours is 2.2.2. Bad things might happen unless you revert torch to 1.x.


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


[Speaker 1] 'SLIE THE GOOD CAN'T MAKE THE ANO THE BAD GET BETTER THE BAD ALWAYS SEEMS TO DRAG DOWN THE GOOD
[Speaker 2] I DON'T WANT TO HAVE TO TO LOOK BACK THIRTY YEARS FROM NOW AND SAY A GOD I WISH I HADN'T DONE THAT LOOKAT IT'S DONE TO MA YOU KNOW


## Test 2: Whisper

In [3]:
#Reference code: https://www.kaggle.com/code/mbmmurad/submission-using-public-finetuned-whisper-model, https://youssefh.substack.com/p/building-and-deploying-a-speech-recognition?r=1sqbmi&utm_campaign=post&utm_medium=web&triedRedirect=true , https://www.kaggle.com/code/brianokechukwu/converting-speech-to-text
import whisper
import soundfile as sf
import os
import librosa
from pyannote.audio import Pipeline

# Load the pre-trained Whisper model
model = whisper.load_model("base")  

# Load the pre-trained speaker diarization model from pyannote
diarization_pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization@2.1")

def speech_to_text_whisper(audio_path):
    # Perform transcription using Whisper
    result = model.transcribe(audio_path)
    return result["text"]

def diarize_and_transcribe(audio_file):
    # Verify if the file exists at the path specified
    if not os.path.exists(audio_file):
        raise ValueError(f"File {audio_file} does not exist")
    
    # Perform speaker diarization using pyannote
    diarization = diarization_pipeline(audio_file)
    
    # Initialize speaker mapping and segments
    speaker_mapping = {}
    speaker_id = 0
    segments = []
    
    # Map speakers to their segments
    for segment, _, speaker in diarization.itertracks(yield_label=True):
        if speaker not in speaker_mapping:
            speaker_mapping[speaker] = f"Speaker {speaker_id + 1}"
            speaker_id += 1

        segments.append({
            "speaker": speaker_mapping[speaker],
            "start": segment.start,
            "end": segment.end
        })

    # Process each segment and transcribe
    for segment in segments:
        start_time = segment["start"]
        end_time = segment["end"]

        # Extract the segment from the audio file
        segment_audio, _ = librosa.load(audio_file, sr=16000, offset=start_time, duration=end_time - start_time)

        # Save the segment as a temporary WAV file
        temp_segment_file = "temp_segment.wav"
        sf.write(temp_segment_file, segment_audio, 16000)

        # Transcribe the segment using Whisper
        transcription = speech_to_text_whisper(temp_segment_file)
        
        # Print the transcription for the current speaker
        print(f"[{segment['speaker']}] {transcription}")
        
        # Optionally, remove the temporary file after transcription
        os.remove(temp_segment_file)

diarize_and_transcribe(audio_file)


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.4.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../.cache/torch/pyannote/models--pyannote--segmentation/snapshots/c4c8ceafcbb3a7a280c2d357aee9fbc9b0be7f9b/pytorch_model.bin`
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


Model was trained with pyannote.audio 0.0.1, yours is 3.1.1. Bad things might happen unless you revert pyannote.audio to 0.x.
Model was trained with torch 1.10.0+cu102, yours is 2.2.2. Bad things might happen unless you revert torch to 1.x.


INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
[NeMo W 2024-12-12 15:17:05 nemo_logging:393] /Applications/anaconda3/lib/python3.12/site-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 ins

[Speaker 1]  It's like the good can't make the, you know, the bad get better. The bad always seems to drag down the good.
[Speaker 2]  I don't want to have to look back 30 years from now and say, God, I wish I hadn't done that. Look what it's done to me, you know.


## Test 3: VOSK

In [4]:
#Reference code: https://stackoverflow.com/questions/69529170/vosk-in-python-get-location-of-transcribed-text-in-audio-file , https://www.kaggle.com/code/vinitkp/speech-recognition , https://www.kaggle.com/code/sohaelshafey/speech-to-text-apis
import wave
import json
import os
from pyannote.audio import Pipeline
from vosk import Model, KaldiRecognizer, SetLogLevel

# Suppress VOSK logs
SetLogLevel(0)

# Path to the VOSK model directory
VOSK_MODEL_PATH = "/Users/laiscoletta/Desktop/DA_Final_Semester/Machine-Learning/Testing Files/vosk-model-small-en-us-0.15"

# Path to the pre-trained pyannote-audio speaker diarization model
DIARIZATION_MODEL_PATH = "pyannote/speaker-diarization@2.1"

# Load the pre-trained speaker diarization model from pyannote
diarization_pipeline = Pipeline.from_pretrained(DIARIZATION_MODEL_PATH)

def perform_diarization(audio_file):
    """
    Perform speaker diarization using pyannote-audio.
    Returns the speaker segments with speaker labels.
    """
    diarization = diarization_pipeline(audio_file)

    # Create a dynamic speaker mapping by assigning names to generic labels
    speaker_mapping = {}
    speaker_segments = []
    speaker_id = 0

    # Process diarization results
    for segment, _, speaker in diarization.itertracks(yield_label=True):
        if speaker not in speaker_mapping:
            speaker_mapping[speaker] = f"Speaker {speaker_id + 1}"  # Dynamically name the speakers
            speaker_id += 1

        speaker_name = speaker_mapping[speaker]
        speaker_segments.append({
            "speaker": speaker_name,
            "start": segment.start,
            "end": segment.end
        })

    return speaker_segments

def transcribe_audio_with_speaker_detection(audio_file, speaker_segments):
    # Open the WAV audio file
    with wave.open(audio_file, "rb") as wf:
        # Validate WAV file format
        if wf.getnchannels() != 1 or wf.getsampwidth() != 2 or wf.getcomptype() != "NONE":
            raise ValueError("Audio file must be WAV format mono PCM.")

        # Load the VOSK model
        model = Model(VOSK_MODEL_PATH)

        # Initialize the recognizer
        recognizer = KaldiRecognizer(model, wf.getframerate())
        recognizer.SetWords(True)

        # Process the audio in chunks
        transcription = []
        speaker_index = 0
        current_segment = speaker_segments[speaker_index] if speaker_segments else None
        current_speaker = current_segment["speaker"] if current_segment else "Unknown"

        while True:
            data = wf.readframes(4000)
            if len(data) == 0:
                break
            if recognizer.AcceptWaveform(data):
                result = json.loads(recognizer.Result())
                text = result.get("text", "")

                # Check if the current segment has ended and change the speaker
                if current_segment and (wf.tell() / wf.getframerate()) >= current_segment["end"]:
                    speaker_index += 1
                    if speaker_index < len(speaker_segments):
                        current_segment = speaker_segments[speaker_index]
                        current_speaker = current_segment["speaker"]

                # Append the transcription with the correct speaker label
                if text:
                    transcription.append(f"{current_speaker}: {text}")

        # Add the final result
        final_result = json.loads(recognizer.FinalResult())
        if final_result.get("text", ""):
            transcription.append(f"{current_speaker}: {final_result['text']}")

    return "\n".join(transcription).strip()

def diarize_and_transcribe(audio_file):
    if not os.path.exists(audio_file):
        raise ValueError(f"File {audio_file} does not exist")
    
    # Perform speaker diarization
    speaker_segments = perform_diarization(audio_file)

    # Perform transcription with speaker labels
    transcription = transcribe_audio_with_speaker_detection(audio_file, speaker_segments)

    return transcription

transcription = diarize_and_transcribe(audio_file)
print(transcription)



Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.4.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../.cache/torch/pyannote/models--pyannote--segmentation/snapshots/c4c8ceafcbb3a7a280c2d357aee9fbc9b0be7f9b/pytorch_model.bin`


Model was trained with pyannote.audio 0.0.1, yours is 3.1.1. Bad things might happen unless you revert pyannote.audio to 0.x.
Model was trained with torch 1.10.0+cu102, yours is 2.2.2. Bad things might happen unless you revert torch to 1.x.


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
LOG (VoskAPI:ReadDataFiles():model.cc:213) Deco

Speaker 1: take the good can't make the in a them bad get better the bad always seems to drag down the good i don't want to have to to look back thirty years now and say god i wish i hadn't done that look good it's done to me know


## Conclusion

- **Wav2Vec 2.0**:
  - [Speaker 1] "SLIE THE GOOD CAN'T MAKE THE ANO THE BAD GET BETTER THE BAD ALWAYS SEEMS TO DRAG DOWN THE GOOD"
  - [Speaker 2] "I DON'T WANT TO HAVE TO TO LOOK BACK THIRTY YEARS FROM NOW AND SAY A GOD I WISH I HADN'T DONE THAT LOOKAT IT'S DONE TO MA YOU KNOW"

- **Whisper**:
  - [Speaker 1] "It's like the good can't make the, you know, the bad get better. The bad always seems to drag down the good."
  - [Speaker 2] "I don't want to have to look back 30 years from now and say, God, I wish I hadn't done that. Look what it's done to me, you know."

- **Vosk**:
  - [Speaker 1] "take the good can't make the in a them bad get better the bad always seems to drag down the good"
  - [Speaker 2] "i don't want to have to to look back thirty years now and say god i wish i hadn't done that look good it's done the me know"

### Observations:
- **Whisper** produced the most accurate and coherent transcription with minimal errors.
- **Wav2Vec 2.0** and **Vosk** had several issues: **Wav2Vec 2.0** was difficult to understand due to unclear words, and **Vosk** had incorrect words, missing phrases, and speaker separation problems.


